# Week 65, Day 3: The Kalman filter: tracking a continuous state

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w065/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The Kalman filter

Complete the predict and update steps of kalman_filter, returning filtered means, covariances and the log-likelihood. Check against building the joint Gaussian over all states and observations explicitly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal as mvn
g0 = np.random.default_rng(653)
T, d, p = 30, 2, 2
A = np.array([[0.95, 0.1], [-0.1, 0.95]])
C = np.array([[1.0, 0.0], [0.3, 1.0]])
Q = 0.05 * np.eye(d); R = 0.4 * np.eye(p)
z = np.zeros((T, d)); y = np.zeros((T, p))
z[0] = g0.multivariate_normal(np.zeros(d), np.eye(d))
y[0] = C @ z[0] + g0.multivariate_normal(np.zeros(p), R)
for t in range(1, T):
    z[t] = A @ z[t - 1] + g0.multivariate_normal(np.zeros(d), Q)
    y[t] = C @ z[t] + g0.multivariate_normal(np.zeros(p), R)

def kalman_filter(y, A, C, Q, R, m0, P0):
    T_, d_ = len(y), len(m0)
    m, P = m0.copy(), P0.copy()
    ms, Ps, ll = [], [], 0.0
    for t in range(T_):
        if t > 0:
            m = A @ m
            P = ...                                     # predicted covariance
        S = C @ P @ C.T + R
        K = ...                                         # Kalman gain
        innov = y[t] - C @ m
        ll += mvn.logpdf(innov, np.zeros(len(S)), S)
        m = m + K @ innov
        P = (np.eye(d_) - K @ C) @ P
        ms.append(m.copy()); Ps.append(P.copy())
    return np.array(ms), np.array(Ps), ll

m_filt, P_filt, ll_kf = kalman_filter(y, A, C, Q, R, np.zeros(d), np.eye(d))

# exact check: build the joint covariance of all states and observations
G = np.zeros((T * d, T * d))
for t in range(T):
    for s in range(t + 1):
        blk = np.linalg.matrix_power(A, t - s)
        G[t * d:(t + 1) * d, s * d:(s + 1) * d] = blk
Qs = [np.eye(d)] + [Q] * (T - 1)
Cov_z = np.zeros((T * d, T * d))
for s in range(T):
    Gs = G[:, s * d:(s + 1) * d]
    Cov_z += Gs @ Qs[s] @ Gs.T
Cbig = np.kron(np.eye(T), C)
Cov_y = Cbig @ Cov_z @ Cbig.T + np.kron(np.eye(T), R)
cross = Cov_z[(T - 1) * d:T * d] @ Cbig.T
m_exact = cross @ np.linalg.solve(Cov_y, y.reshape(-1))
ll_direct = mvn.logpdf(y.reshape(-1), np.zeros(T * p), Cov_y)
print(f"filtered estimate at the last step {np.round(m_filt[-1], 4)}, exact {np.round(m_exact, 4)}")
print(f"log-likelihood from innovations {ll_kf:.6f}, direct {ll_direct:.6f}")

In [ ]:
import numpy as np
def test_exact():
    "the filtered estimate at the last step matches exact Gaussian conditioning on the whole sequence up to that point"
    assert np.allclose(m_filt[-1], m_exact, atol=1e-8)
def test_loglik():
    "the summed innovation likelihoods match the Gaussian log-density of the whole observation sequence"
    assert abs(ll_kf - ll_direct) < 1e-8

test_exact()
test_loglik()
print("All checks passed")

### Exercise: Misstating the noise

Simulate a scalar random walk observed with noise of variance 1. Run the filter with the measurement noise set to 0.1, 1 and 10 times the truth, and measure the roughness of each estimate (mean squared step) and the coverage of its 95% intervals.

In [ ]:
gg = np.random.default_rng(6531)
T2 = 2000
zs = np.zeros(T2)
for t in range(1, T2):
    zs[t] = 0.99 * zs[t - 1] + np.sqrt(0.1) * gg.normal()
ys = zs + gg.normal(size=T2)
rough, cover = {}, {}
for fac in (0.1, 1.0, 10.0):
    m, P = 0.0, 1.0
    est, sd = [], []
    for t in range(T2):
        m, P = 0.99 * m, 0.99 ** 2 * P + 0.1
        K = P / (P + 1.0 * fac)
        m = m + K * (ys[t] - m); P = (1 - K) * P
        est.append(m); sd.append(np.sqrt(P))
    est, sd = np.array(est), np.array(sd)
    rough[fac] = np.mean(np.diff(est) ** 2)
    cover[fac] = ...
    print(f"assumed measurement noise {fac:4.1f} times the truth: roughness {rough[fac]:.4f}, coverage {cover[fac]:.3f}")

In [ ]:
import numpy as np
def test_smoother():
    "overstating the measurement noise gives a smoother estimate"
    assert rough[10.0] < rough[1.0] < rough[0.1]
def test_cover():
    "only the correct noise gives well-calibrated intervals"
    assert abs(cover[1.0] - 0.95) < 0.05 and cover[0.1] < 0.9

test_smoother()
test_cover()
print("All checks passed")

### Exercise: A Kalman decoder

Simulate 600 bins of a reaching cursor (position and velocity in two dimensions) and 12 noisy neurons tuned to velocity. Decode with a Kalman filter and compare with bin-by-bin linear regression of velocity on firing rates.

In [ ]:
gc = np.random.default_rng(6532)
dt = 0.05
A4 = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 0.93, 0], [0, 0, 0, 0.93]])
Q4 = np.diag([1e-6, 1e-6, 0.02, 0.02])
N = 12
pref = gc.uniform(0, 2 * np.pi, N)
C4 = np.column_stack([np.zeros(N), np.zeros(N), 3 * np.cos(pref), 3 * np.sin(pref)])
R4 = np.diag(gc.uniform(2.0, 5.0, N))
Tz = 600
state = np.zeros((Tz, 4)); y_neural = np.zeros((Tz, N))
for t in range(1, Tz):
    state[t] = A4 @ state[t - 1] + gc.multivariate_normal(np.zeros(4), Q4)
    y_neural[t] = C4 @ state[t] + gc.multivariate_normal(np.zeros(N), R4)

m_dec, P_dec, _ = kalman_filter(y_neural, A4, C4, Q4, R4, np.zeros(4), np.eye(4))
W = np.linalg.lstsq(y_neural, state[:, 2:], rcond=None)[0]     # bin-by-bin regression (cheating: fitted on all data)
reg = y_neural @ W
corr_kf = ...
corr_reg = np.mean([np.corrcoef(reg[:, k], state[:, 2 + k])[0, 1] for k in (0, 1)])
print(f"velocity decoding correlation: Kalman {corr_kf:.3f}, bin-by-bin regression {corr_reg:.3f}")
plt.plot(state[:200, 2], "k", label="true velocity x"); plt.plot(m_dec[:200, 2], label="Kalman"); plt.plot(reg[:200, 0], alpha=0.5, label="regression"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_better():
    "the Kalman decoder tracks velocity better than regressing each bin independently"
    assert corr_kf > corr_reg + 0.05
def test_online():
    "the decoder uses only past and present bins"
    import dis
    src = dis.Bytecode(kalman_filter).codeobj.co_consts
    assert m_dec.shape == (len(y_neural), 4)

test_better()
test_online()
print("All checks passed")